### 1.问题
智能体工作越久，messages数组越胖。  
每次读文件、跑命令的输出都永久留在上下文里。
### 2.解决方法
1. 父智能体有一个task工具。子智能体拥有除了task以外的所有基础工具（禁止递归生成）。
```python
PARENT_TOOLS = CHILD_TOOLS + [
    {
        "type": "function",
        "function": {
            "name": "task",
            "description": "Spawn a subagent with fresh context."
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {
                        "type": "string",
                    },
                    "required": ["prompt"],
                }
            }
        }
    }
]
```
2. 子智能体以message=[]启动，运行自己的循环。只有最终文本返回给父智能体。
```python
import json

def run_subagent(prompt: str) -> str:
    sub_messages = [
        {"role": "system", "content": SUBAGENT_SYSTEM},
        {"role": "user", "content": prompt},
    ]
    response = None

    for _ in range(30):
        response = client.chat.completions.create(
            model=MODEL,
            messages=sub_messages,
            tools=CHILD_TOOLS,
            max_tokens=8000,
        )
        message = response.choices[0].message
        sub_messages.append(message.model_dump())

        if not message.tool_calls:
            break

        for tool_call in message.tool_calls:
            handler = TOOL_HANDLERS.get(tool_call.function.name)
            output = (
                handler(**json.loads(tool_call.function.arguments))
                if handler
                else f"Unknown tool: {tool_call.function.name}"
            )
            sub_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(output)[:50000],
            })
    else:
        response = client.chat.completions.create(
            model=MODEL,
            messages=sub_messages,
            tools=CHILD_TOOLS,
            max_tokens=8000,
        )

    return response.choices[0].message.content or "(no summary)"
```
子智能体可能跑了30+次工具调用，但整个消息历史直接丢弃。父智能体收到的只是一段摘要文本，作为普通的tool_result返回。

In [ ]:
import os
import subprocess
import json

from openai import OpenAI
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

WORKDIR = Path.cwd()
client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"))

def safe_path(p: str) -> Path:
    path = (WORKDIR / p).resolve()
    if not path.is_relative_to(WORKDIR):
        raise ValueError(f"Path escapes workspace: {p}")
    return path

def run_bash(command: str) -> str:
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return f"Error: Dangerous command blocked."
    try:
        r = subprocess.run(command, shell=True, cwd=WORKDIR, capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        return out[:50000] if out else "(no output)"
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)."

def run_read(path: str, limit: int = None) -> str:
    try:
        lines = safe_path(path).read_text(encoding="utf-8").splitlines()
        if limit and limit < len(lines):
            lines = lines[:limit] + [f"... ({len(lines) - limit} more)"]
        return "\n".join(lines)[:50000]
    except Exception as e:
        return f"Error: {e}"

def run_write(path: str, content: str) -> str:
    try:
        fp = safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content, encoding="utf-8")
        return f"Wrote {len(content)} bytes."
    except Exception as e:
        return f"Error: {e}"

def run_edit(path: str, old_text: str, new_text: str) -> str:
    try:
        fp = safe_path(path)
        content = fp.read_text(encoding="utf-8")


